In [1]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import torchaudio


In [2]:
# ===== PATHS =====
DATA_DIR  = "DATA"
VIDEO_DIR = os.path.join(DATA_DIR, "raw_videos")
IMAGE_DIR = os.path.join(DATA_DIR, "images")
AUDIO_DIR = os.path.join(DATA_DIR, "audio")

print(os.path.exists(VIDEO_DIR), os.path.exists(IMAGE_DIR), os.path.exists(AUDIO_DIR))


True True True


In [3]:
emotion_map = {
    "01": 0,  # neutral
    "02": 1,  # calm
    "03": 2,  # happy
    "04": 3,  # sad
    "05": 4,  # angry
    "06": 5,  # fearful
    "07": 6,  # disgust
    "08": 7   # surprised
}

idx_to_emotion = {v: k for k, v in emotion_map.items()}


In [4]:
records = []

for root, _, files in os.walk(VIDEO_DIR):
    for f in files:

        if not f.lower().endswith(".mp4"):
            continue

        parts = f.replace(".mp4", "").split("-")

        # strict RAVDESS filename check
        if len(parts) != 7:
            continue

        # keep ONLY full audio-video
        if parts[0] != "01":
            continue

        image_path = os.path.join(IMAGE_DIR, f.replace(".mp4", ".jpg"))
        audio_path = os.path.join(AUDIO_DIR, f.replace(".mp4", ".wav"))

        if not (os.path.exists(image_path) and os.path.exists(audio_path)):
            continue

        records.append({
            "video": os.path.join(root, f),
            "image": image_path,
            "audio": audio_path,
            "emotion": emotion_map[parts[2]],
            "actor": int(parts[-1])
        })

df = pd.DataFrame(records)
len(df)


1440

In [5]:
train_df = df[df["actor"] <= 20].reset_index(drop=True)
val_df   = df[df["actor"] > 20].reset_index(drop=True)

len(train_df), len(val_df)


(1200, 240)

In [6]:
image_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5],
                std=[0.5, 0.5, 0.5])
])


In [7]:
class EmotionDataset(Dataset):
    def __init__(self, df, image_transform=None):
        self.df = df
        self.image_transform = image_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # ---- Image ----
        image = Image.open(row["image"]).convert("RGB")
        if self.image_transform:
            image = self.image_transform(image)

        # ---- Audio ----
        waveform, sr = torchaudio.load(row["audio"])
        if sr != 16000:
            waveform = torchaudio.functional.resample(waveform, sr, 16000)

        label = torch.tensor(row["emotion"], dtype=torch.long)

        return image, waveform, label


In [8]:
train_ds = EmotionDataset(train_df, image_transform)
val_ds   = EmotionDataset(val_df, image_transform)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=16, shuffle=False)


In [9]:
img, wav, label = train_ds[0]

img.shape      # [3, 224, 224]
wav.shape      # [1, T]
label


tensor(0)